# Milestone 18 Objective

Integrate the already validated UI Testing Agent into the main LangGraph workflow.

## Why Integration Comes After Standalone Validation

The UI Testing Agent first worked alone through `START -> ui_testing -> END`. This milestone connects it only after its Selenium result format, screenshot metadata, and failure classification are stable.

## Integrated Workflow

`START -> orchestrator -> repo_analyzer -> rag -> test_planner -> api_testing -> ui_testing -> bug_analysis -> report -> END`

## State Handoff

- Test Planner writes `ui_tests` inside `test_plan`.
- UI Agent reads `ui_tests` and `target_url`.
- UI Agent writes `ui_results`, `ui_result_path`, and `screenshots`.
- Bug Analysis reads UI results and classifies UI anomalies.
- Report displays UI summary, UI rows, and screenshot paths.

## Safety Note

The UI Agent does not start Django, run Docker, or execute target repository code. If the browser or target app is unavailable, it records `environment_error` and the workflow can still produce a report.

Target repository: https://github.com/Vitaee/DjangoRestAPI

Target URL: http://localhost:8000

In [1]:
# Part A - fake repo integration with API and UI execution mocked
from pathlib import Path
from tempfile import TemporaryDirectory

from test_auto.agents import api_testing_agent, ui_testing_agent
from test_auto.graph.workflow import run_workflow

def write_file(root: Path, relative: str, content: str):
    path = root / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')

def make_fake_repo(root: Path) -> Path:
    repo = root / 'fake_django_rest_repo'
    write_file(repo, 'README.md', '# Todo API\nJWT authentication protects Todo CRUD routes.\n')
    write_file(repo, 'requirements.txt', 'django\ndjangorestframework\n')
    write_file(repo, 'todo/urls.py', 'from django.urls import path\nurlpatterns = [path("api/todos/", lambda r: None)]\n')
    write_file(repo, 'todo/views.py', 'def todo_list(request): pass\n')
    write_file(repo, 'templates/login.html', '<form><input name="username"><input type="password"></form>')
    return repo

def fake_api(target_url, test_case, **kwargs):
    expected = test_case.get('expected_status') or 200
    return {
        'id': test_case.get('id'), 'name': test_case.get('name'), 'method': test_case.get('method'),
        'endpoint': test_case.get('endpoint'), 'status': 'passed', 'expected_status': expected,
        'actual_status': expected, 'duration_ms': 5.0, 'details': 'mocked API execution',
        'evidence': {}, 'assertions': [{'type': 'status_code', 'passed': True}], 'error_type': None,
    }

def fake_ui(target_url, test_case, run_id, discovered_ui_flows=None, user_preferences=None):
    return {
        'id': test_case.get('id'), 'name': test_case.get('name'), 'flow': test_case.get('flow'),
        'status': 'assertion_error', 'target_path': '/login/', 'target_url': target_url.rstrip('/') + '/login/',
        'duration_ms': 6.0, 'details': 'mocked missing login form',
        'screenshot': {'path': f'results/runs/{run_id}/screenshots/UI_001_assertion.png', 'reason': 'assertion', 'created': True},
        'assertions': [{'type': 'login_form_present', 'passed': False}], 'error_type': 'assertion_error',
        'evidence': {'title': 'Login'},
    }

api_testing_agent.execute_api_test_case = fake_api
ui_testing_agent.execute_ui_test_case = fake_ui

with TemporaryDirectory() as tmp:
    repo = make_fake_repo(Path(tmp))
    final_state = run_workflow({
        'repo_path': str(repo),
        'target_url': 'http://localhost:8000',
        'user_preferences': {
            'test_types': ['api', 'ui'],
            'execution_mode': 'sequential',
            'focus': 'JWT authentication todo CRUD API tests',
            'planner_use_llm': False,
        },
        'errors': [],
        'agent_logs': [],
    })
    compact = {
        'run_id': final_state.get('run_id'),
        'ui_summary': (final_state.get('ui_results') or {}).get('summary'),
        'bug_summary': (final_state.get('bug_results') or {}).get('summary'),
        'report_html_path': final_state.get('report_html_path'),
    }
compact

{'run_id': 'run_20260520T084258Z_17d2df36',
 'ui_summary': {'total_tests': 1,
  'passed': 0,
  'failed': 1,
  'skipped': 0,
  'errors': 0,
  'pass_rate': 0.0},
 'bug_summary': {'total_anomalies': 1,
  'high': 0,
  'medium': 1,
  'low': 0,
  'info': 0,
  'by_classification': {'assertion_error': 1}},
 'report_html_path': 'reports\\generated\\report_run_20260520T084258Z_17d2df36.html'}

In [2]:
# Part B - inspect UI anomaly evidence from the previous run
[
    anomaly for anomaly in (final_state.get('bug_results') or {}).get('anomalies', [])
    if anomaly.get('source_agent') == 'ui_testing'
]

[{'id': 'BUG_UI_001',
  'type': 'ui_assertion_failed',
  'severity': 'medium',
  'source_agent': 'ui_testing',
  'classification': 'assertion_error',
  'title': 'UI assertion failed: login_flow_is_visible',
  'evidence': {'source_agent': 'ui_testing',
   'test_id': 'UI_001',
   'test_name': 'login_flow_is_visible',
   'method': None,
   'endpoint': None,
   'flow': 'login',
   'target_path': '/login/',
   'target_url': 'http://localhost:8000/login/',
   'expected_status': None,
   'actual_status': None,
   'status': 'assertion_error',
   'details': 'mocked missing login form',
   'duration_ms': 6.0,
   'screenshot_path': 'results/runs/run_20260520T084258Z_17d2df36/screenshots/UI_001_assertion.png',
   'evidence_path': 'results\\runs\\run_20260520T084258Z_17d2df36\\ui_result.json'},
  'recommendation': 'Verify expected UI text/form behavior against the application requirements.',
  'confidence': 0.75}]

## Optional Real Target App

This cell requires the Django target app running at `http://localhost:8000` and a browser available. Do not start the app from this notebook.

In [3]:
# Optional: load a previous test_plan.json and run the integrated workflow against the real target.
# from pathlib import Path
# import json
# test_plan = json.loads(Path('results/runs/<run_id>/test_plan.json').read_text(encoding='utf-8'))
# Display generated report path and UI summary after a real authorized run.

In [4]:
# Part C - graph visualization
from test_auto.graph.workflow import build_graph

graph = build_graph()
try:
    graph.get_graph().draw_mermaid_png()
except Exception:
    print('START -> orchestrator -> repo_analyzer -> rag -> test_planner -> api_testing -> ui_testing -> bug_analysis -> report -> END')